## **Random Forest STL na reprezentacji fingerprint**

Wykorzystana reprezentacja: **ECFP4**

Lista endpointów:


1. Caco-2 (Wang)
2. Lipophilicity (AstraZeneca)
3. Solubility (AqSolDB)
4. HIA (Hou)
5. Half Life (Obach)
6. Clearance Hepatocyte (AstraZeneca)
7. CYP3A4 Inhibition (Veith)
8. VDss (Lombardo)
9. AMES Mutagenicity
10. hERG (Wang)

Metryki jakości: AUROC (klasyfikacja), RMSE(regresja), Accuracy, F1
(klasyfikacja), R^2(regresja), MAE(regresja)

In [ ]:
!pip install rdkit pandas numpy scikit-learn -U
!pip install pytdc --no-dependencies


In [ ]:
!pip install fuzzywuzzy

Krok 1: Przygotowanie danych i zamiana na fingerprinty

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem

class Featurizer:
    def __init__(self, y_column, smiles_col='Drug', **kwargs):
        self.y_column = y_column
        self.smiles_col = smiles_col
        self.__dict__.update(kwargs)

    def __call__(self, df):
        raise NotImplementedError()
class ECFPFeaturizer(Featurizer): #
    def __init__(self, y_column, radius=2, length=1024, **kwargs):
        self.radius = radius
        self.length = length
        super().__init__(y_column, **kwargs)

    def __call__(self, df):
        fingerprints = []
        labels = []
        for i, row in df.iterrows():
            y = row[self.y_column]
            smiles = row[self.smiles_col]
            mol = Chem.MolFromSmiles(smiles)
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, self.radius, nBits=self.length)
            fingerprints.append(fp)
            labels.append(y)
        fingerprints = np.array(fingerprints)
        labels = np.array(labels)
        return fingerprints, labels

Funkcje pomocnicze do zapisywania i wczytywania danych oraz wypisywania metryk

In [ ]:
import pickle

def split_and_save(tdc_data, dataset_name):
    split    = tdc_data.get_split(seed=42, frac=[0.8, 0.0, 0.2])
    train_df = split["train"]
    test_df  = split["test"]

    filepath = f"{dataset_name}_split.pkl"
    with open(filepath, "wb") as f:
        pickle.dump({"train": train_df, "test": test_df}, f)

    return train_df, test_df

In [ ]:
import pickle

def load_split_pickle(dataset_name):
    filepath = f"{dataset_name}_split.pkl"

    with open(filepath, "rb") as f:
        split = pickle.load(f)

    return split["train"], split["test"]

In [ ]:
def print_metrics(metrics, task='classification'):
    print(f"\n{'='*40}")
    print(f"Best params: {metrics['best_params']}")
    print(f"{'='*40}")
    if task == 'classification':
        print(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}")
        print(f"  F1       : {metrics['test_metrics']['f1']:.4f}")
        print(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}")
    else:
        print(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}")
        print(f"  MAE      : {metrics['test_metrics']['mae']:.4f}")
        print(f"  R²       : {metrics['test_metrics']['r2']:.4f}")
    print(f"{'='*40}\n")


def save_metrics(metrics, dataset_name, filepath, task='classification'):
    with open(filepath, 'a') as f:
        f.write(f"\n{'='*40}\n")
        f.write(f"Endpoint    : {dataset_name}\n")
        f.write(f"{'='*40}\n")
        f.write(f"Best params: {metrics['best_params']}\n")
        f.write(f"{'-'*40}\n")
        if task == 'classification':
            f.write(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}\n")
            f.write(f"  F1       : {metrics['test_metrics']['f1']:.4f}\n")
            f.write(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}\n")
        else:
            f.write(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}\n")
            f.write(f"  MAE      : {metrics['test_metrics']['mae']:.4f}\n")
            f.write(f"  R²       : {metrics['test_metrics']['r2']:.4f}\n")
        f.write(f"{'='*40}\n")

Krok 2: Trening Random Forest

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def train_rf_regression(X_train, X_test, y_train, y_test, rf_param_grid=None, cv_folds=5):

    if rf_param_grid is None:
        rf_param_grid = {
            "n_estimators": [200, 500, 1000, 1500],
            "max_depth": [None, 10,15,20,25,30],
            "min_samples_split": [2, 5,10],
            "max_features": ["sqrt", "log2"]

        }

    n_iter = 1
    for values in rf_param_grid.values():
        n_iter *= len(values)

    random_search = RandomizedSearchCV(
        estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
        param_distributions=rf_param_grid,
        cv=cv_folds,
        n_iter=n_iter,
        scoring='neg_root_mean_squared_error',
        n_jobs=-1,
        random_state=42
    )
    random_search.fit(X_train, y_train)
    best_params = random_search.best_params_

    final_model = random_search.best_estimator_

    y_test_pred = final_model.predict(X_test)

    metrics = {
        "best_params": best_params,
        "test_metrics": {
            "rmse": np.sqrt(mean_squared_error(y_test, y_test_pred)),
            "mae":  mean_absolute_error(y_test, y_test_pred),
            "r2":   r2_score(y_test, y_test_pred)
        },
        "model": final_model
    }

    return metrics

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

def train_rf_classification(X_train, X_test, y_train, y_test, rf_param_grid=None, cv_folds=5):

    if rf_param_grid is None:
        rf_param_grid = {
          "n_estimators": [200, 500, 1000, 1500],
            "max_depth": [None, 10,15,20,25,30],
            "min_samples_split": [2, 5,10],
            "max_features": ["sqrt", "log2"]
        }

    n_iter = 1
    for values in rf_param_grid.values():
        n_iter *= len(values)

    random_search = RandomizedSearchCV(
        estimator=RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced"),
        param_distributions=rf_param_grid,
        cv=cv_folds,
        n_iter=n_iter,
        scoring='roc_auc',
        n_jobs=-1,
        random_state=42
    )
    random_search.fit(X_train, y_train)
    best_params = random_search.best_params_

    final_model = random_search.best_estimator_

    y_test_pred      = final_model.predict(X_test)
    y_test_pred_prob = final_model.predict_proba(X_test)[:, 1]

    metrics = {
        "best_params": best_params,
        "test_metrics": {
            "accuracy": accuracy_score(y_test, y_test_pred),
            "f1":       f1_score(y_test, y_test_pred),
            "auroc":    roc_auc_score(y_test, y_test_pred_prob),
        },
        "model": final_model
    }

    return metrics

Endpoint 1: Caco-2 (Wang)

In [ ]:
from tdc.single_pred import ADME

data = ADME(name = 'Caco2_Wang')

train, test = split_and_save(data, 'Caco2_Wang')

featurizer = ECFPFeaturizer(y_column="Y", radius=2, length=1024)

X_train, y_train = featurizer(train)
X_test,  y_test  = featurizer(test)

metrics = train_rf_regression(X_train, X_test, y_train, y_test)

print_metrics(metrics, task="regression")
save_metrics(metrics, "Caco2_Wang", "metrics.txt", task="regression")


Downloading...
100%|██████████| 82.5k/82.5k [00:00<00:00, 1.66MiB/s]
Loading...
Done!
[12:46:48] DEPRECATION WARNING: please use MorganGenerator
[12:46:48] DEPRECATION WARNING: please use MorganGenerator
[12:46:48] DEPRECATION WARNING: please use MorganGenerator
[12:46:48] DEPRECATION WARNING: please use MorganGenerator
[12:46:48] DEPRECATION WARNING: please use MorganGenerator
[12:46:48] DEPRECATION WARNING: please use MorganGenerator
[12:46:48] DEPRECATION WARNING: please use MorganGenerator
[12:46:48] DEPRECATION WARNING: please use MorganGenerator
[12:46:48] DEPRECATION WARNING: please use MorganGenerator
[12:46:48] DEPRECATION WARNING: please use MorganGenerator
[12:46:48] DEPRECATION WARNING: please use MorganGenerator
[12:46:48] DEPRECATION WARNING: please use MorganGenerator
[12:46:48] DEPRECATION WARNING: please use MorganGenerator
[12:46:48] DEPRECATION WARNING: please use MorganGenerator
[12:46:48] DEPRECATION WARNING: please use MorganGenerator
[12:46:48] DEPRECATION WARNIN

KeyboardInterrupt: 

Endpoint2: Lipophilicity

In [ ]:
from tdc.single_pred import ADME

data = ADME(name = 'Lipophilicity_AstraZeneca')

train, test = split_and_save(data, 'Lipophilicity_AstraZeneca')

featurizer = ECFPFeaturizer(y_column="Y", radius=2, length=1024)

X_train, y_train = featurizer(train)
X_test,  y_test  = featurizer(test)

metrics = train_rf_regression(X_train, X_test, y_train, y_test)

print_metrics(metrics, task="regression")
save_metrics(metrics, 'Lipophilicity_AstraZeneca', "metrics.txt", task="regression")

Endpoint 3: Solubility

In [ ]:
from tdc.single_pred import ADME

data = ADME(name = 'Solubility_AqSolDB')

train, test = split_and_save(data, 'Solubility_AqSolDB')

featurizer = ECFPFeaturizer(y_column="Y", radius=2, length=1024)

X_train, y_train = featurizer(train)
X_test,  y_test  = featurizer(test)

metrics = train_rf_regression(X_train, X_test, y_train, y_test)

print_metrics(metrics, task="regression")
save_metrics(metrics, 'Solubility_AqSolDB', "metrics.txt", task="regression")

Endpoint 4: HIA

In [ ]:
from tdc.single_pred import ADME

data = ADME(name = 'HIA_Hou')

train, test = split_and_save(data, 'HIA_Hou')

featurizer = ECFPFeaturizer(y_column="Y", radius=2, length=1024)

X_train, y_train = featurizer(train)
X_test,  y_test  = featurizer(test)

metrics = train_rf_classification(X_train, X_test, y_train, y_test)

print_metrics(metrics)
save_metrics(metrics, 'HIA_Hou', "metrics.txt")

Endpoint 5: Half Life

In [ ]:
from tdc.single_pred import ADME

data = ADME(name = 'Half_Life_Obach')

train, test = split_and_save(data, 'Half_Life_Obach')

featurizer = ECFPFeaturizer(y_column="Y", radius=2, length=1024)

X_train, y_train = featurizer(train)
X_test,  y_test  = featurizer(test)

metrics = train_rf_regression(X_train, X_test, y_train, y_test)

print_metrics(metrics, task="regression")
save_metrics(metrics, 'Half_Life_Obach', "metrics.txt", task="regression")

Endpoint 6: Clearance Hepatocyte

In [ ]:
from tdc.single_pred import ADME

data = ADME(name = 'Clearance_Hepatocyte_AZ')

train, test = split_and_save(data, 'Clearance_Hepatocyte_AZ')

featurizer = ECFPFeaturizer(y_column="Y", radius=2, length=1024)

X_train, y_train = featurizer(train)
X_test,  y_test  = featurizer(test)

metrics = train_rf_regression(X_train, X_test, y_train, y_test)

print_metrics(metrics, task="regression")
save_metrics(metrics, "Clearance_Hepatocyte_AZ", "metrics.txt", task="regression")

Endpoint 7: CYP3A4 Inhibition

In [ ]:
from tdc.single_pred import ADME

data = ADME(name = 'CYP3A4_Veith')

train, test = split_and_save(data, 'CYP3A4_Veith')

featurizer = ECFPFeaturizer(y_column="Y", radius=2, length=1024)

X_train, y_train = featurizer(train)
X_test,  y_test  = featurizer(test)

metrics = train_rf_classification(X_train, X_test, y_train, y_test)

print_metrics(metrics)
save_metrics(metrics, 'CYP3A4_Veith', "metrics.txt")

Endpoint 8: VDss

In [ ]:
from tdc.single_pred import ADME

data = ADME(name = 'VDss_Lombardo')

train, test = split_and_save(data, 'VDss_Lombardo')

featurizer = ECFPFeaturizer(y_column="Y", radius=2, length=1024)

X_train, y_train = featurizer(train)
X_test,  y_test  = featurizer(test)

metrics = train_rf_regression(X_train, X_test, y_train, y_test)

print_metrics(metrics, task="regression")
save_metrics(metrics, 'VDss_Lombardo', "metrics.txt", task="regression")

Endpoint 9: AMES Mutagenicity

In [ ]:
from tdc.single_pred import Tox

data = Tox(name = 'AMES')

train, test = split_and_save(data, 'AMES')

featurizer = ECFPFeaturizer(y_column="Y", radius=2, length=1024)

X_train, y_train = featurizer(train)
X_test,  y_test  = featurizer(test)

metrics = train_rf_classification(X_train, X_test, y_train, y_test)

print_metrics(metrics)
save_metrics(metrics, 'AMES', "metrics.txt")

Endpoint 10: hERG (Wang)

In [ ]:
from tdc.single_pred import Tox

data = Tox(name = 'hERG')

train, test = split_and_save(data, 'hERG')

featurizer = ECFPFeaturizer(y_column="Y", radius=2, length=1024)

X_train, y_train = featurizer(train)
X_test,  y_test  = featurizer(test)

metrics = train_rf_classification(X_train, X_test, y_train, y_test)

print_metrics(metrics)
save_metrics(metrics, 'hERG', "metrics.txt")